In [4]:
# This cell WRITES the streamlit app to a .py file
# We can't run streamlit inside jupyter, so we save it first

app_code = '''
import streamlit as st
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np

# ============================================================
# PAGE CONFIG
# ============================================================
st.set_page_config(
    page_title="Pneumonia Detector",
    page_icon="🫁",
    layout="centered"
)

# ============================================================
# DEFINE BOTH MODEL ARCHITECTURES
# (must match exactly what we trained)
# ============================================================

class PneumoniaCNN(nn.Module):
    def __init__(self):
        super(PneumoniaCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


def load_resnet():
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(
        nn.Linear(2048, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 2)
    )
    return model


# ============================================================
# LOAD MODELS (cached so they only load once)
# ============================================================

@st.cache_resource
def load_models():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # CNN from scratch
    cnn = PneumoniaCNN()
    cnn.load_state_dict(torch.load("best_cnn_scratch.pth", map_location=device))
    cnn.to(device).eval()

    # ResNet-50
    resnet = load_resnet()
    resnet.load_state_dict(torch.load("best_resnet50.pth", map_location=device))
    resnet.to(device).eval()

    return cnn, resnet, device

# ============================================================
# IMAGE PREPROCESSING
# ============================================================

def preprocess(image):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return transform(image).unsqueeze(0)  # add batch dimension

# ============================================================
# PREDICT FUNCTION
# ============================================================

def predict(model, image_tensor, device):
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        output = model(image_tensor)
        probs  = torch.softmax(output, dim=1)[0]
        pred   = output.argmax(dim=1).item()
    return pred, probs[0].item(), probs[1].item()

# ============================================================
# UI — HEADER
# ============================================================

st.markdown("""
    <h1 style="text-align:center; color:#1F4E79;">
        🫁 Pneumonia Detection System
    </h1>
    <p style="text-align:center; color:#555; font-size:16px;">
        Upload a chest X-ray image to detect Pneumonia using Deep Learning
    </p>
    <hr style="border: 1px solid #1F4E79;">
""", unsafe_allow_html=True)

# ============================================================
# SIDEBAR — MODEL SELECTION & INFO
# ============================================================

st.sidebar.title("⚙️ Settings")
model_choice = st.sidebar.radio(
    "Choose Model:",
    ["ResNet-50 (Recommended)", "CNN from Scratch"]
)

st.sidebar.markdown("---")
st.sidebar.markdown("### 📊 Model Performance")
st.sidebar.markdown("""
| Metric | CNN | ResNet |
|--------|-----|--------|
| Accuracy | 83.49% | 83.65% |
| ROC-AUC | 0.914 | 0.963 |
| Pneumonia Recall | 96% | 99% |
| Missed Sick | 16 | **2** |
""")

st.sidebar.markdown("---")
st.sidebar.markdown("### ℹ️ About")
st.sidebar.markdown("""
**Project:** ANN & Deep Learning  
**Semester:** BS AI — 6th  
**Team:**  
- Inam-Ur-Rehman (076)  
- M. Abdullah Nazir (078)  
**Supervisor:** Dr Muhammad Gufran Khan
""")

# ============================================================
# MAIN — FILE UPLOAD
# ============================================================

st.markdown("### 📤 Upload Chest X-Ray Image")
uploaded_file = st.file_uploader(
    "Drag and drop or click to upload",
    type=["jpg", "jpeg", "png"],
    help="Upload a frontal chest X-ray image (JPEG or PNG)"
)

if uploaded_file is not None:

    # Show the uploaded image
    image = Image.open(uploaded_file).convert("RGB")
    col1, col2 = st.columns([1, 1])

    with col1:
        st.markdown("#### 🖼️ Uploaded X-Ray")
        st.image(image, use_column_width=True, caption="Input X-Ray Image")

    with col2:
        st.markdown("#### 🔬 Analysis Result")

        with st.spinner("Analyzing X-ray... please wait"):
            # Load models
            cnn_model, resnet_model, device = load_models()

            # Preprocess
            img_tensor = preprocess(image)

            # Choose model
            if "ResNet" in model_choice:
                pred, prob_normal, prob_pneumonia = predict(resnet_model, img_tensor, device)
                model_name = "ResNet-50"
            else:
                pred, prob_normal, prob_pneumonia = predict(cnn_model, img_tensor, device)
                model_name = "CNN from Scratch"

        # ---- RESULT DISPLAY ----
        if pred == 1:
            st.error("🚨 PNEUMONIA DETECTED")
            st.markdown(f"""
            <div style="background:#ffe6e6; padding:20px; border-radius:10px;
                        border-left: 6px solid #e74c3c;">
                <h3 style="color:#c0392b;">⚠️ Pneumonia Detected</h3>
                <p style="font-size:15px;">
                    The model has identified signs of <b>pneumonia</b>
                    in this chest X-ray.<br><br>
                    <b>Please consult a qualified radiologist or physician
                    for professional medical advice.</b>
                </p>
            </div>
            """, unsafe_allow_html=True)
        else:
            st.success("✅ NORMAL — No Pneumonia Detected")
            st.markdown(f"""
            <div style="background:#e6ffe6; padding:20px; border-radius:10px;
                        border-left: 6px solid #27ae60;">
                <h3 style="color:#1e6b3c;">✅ Lungs Appear Normal</h3>
                <p style="font-size:15px;">
                    No signs of pneumonia were detected in this X-ray.<br><br>
                    <b>This is an AI tool — always confirm with a
                    medical professional.</b>
                </p>
            </div>
            """, unsafe_allow_html=True)

        # ---- CONFIDENCE BARS ----
        st.markdown("#### 📊 Confidence Scores")
        st.markdown(f"**Model used:** {model_name}")

        st.markdown("🟢 Normal")
        st.progress(prob_normal)
        st.markdown(f"**{prob_normal*100:.1f}%** confidence it is Normal")

        st.markdown("🔴 Pneumonia")
        st.progress(prob_pneumonia)
        st.markdown(f"**{prob_pneumonia*100:.1f}%** confidence it is Pneumonia")

    # ---- DISCLAIMER ----
    st.markdown("---")
    st.warning("""
    ⚠️ **Medical Disclaimer:** This tool is developed for academic purposes only.
    It is NOT a substitute for professional medical diagnosis.
    Always consult a qualified radiologist or physician for medical decisions.
    """)

else:
    # Instructions when no file uploaded
    st.info("👆 Please upload a chest X-ray image above to get started.")

    st.markdown("### 💡 How to use:")
    st.markdown("""
    1. **Select a model** from the sidebar (ResNet-50 recommended)
    2. **Upload** a chest X-ray image (JPG or PNG)
    3. **Wait** a few seconds for analysis
    4. **View** the prediction and confidence scores
    """)

    st.markdown("### 🧪 Sample X-rays for testing:")
    st.markdown("""
    You can use any image from your test dataset:

SyntaxError: invalid syntax (746053963.py, line 4)

In [ ]:
import os

app_path = "pneumonia_app.py"

if os.path.exists(app_path):
    size = os.path.getsize(app_path)
    print(f"✅ File exists: {app_path}")
    print(f"   Size: {size:,} bytes")
    print(f"\n✅ Also check these model files exist:")
    for f in ["best_cnn_scratch.pth", "best_resnet50.pth"]:
        exists = os.path.exists(f)
        size   = os.path.getsize(f) if exists else 0
        print(f"   {f}: {'✅ Found' if exists else '❌ MISSING'}  ({size/1024/1024:.1f} MB)")
else:
    print("❌ File not found — re-run Cell 1")

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           HOW TO LAUNCH YOUR WEBSITE                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. Open a NEW Command Prompt window                         ║
║                                                              ║
║  2. Type these commands:                                     ║
║                                                              ║
║  cd "C:\\Users\\SS\\OneDrive\\Desktop\\University\\Semester 6\\pneumonia_project"
║  venv\\Scripts\\activate                                      ║
║  streamlit run pneumonia_app.py                              ║
║                                                              ║
║  3. Your browser will open automatically at:                 ║
║     http://localhost:8501                                    ║
║                                                              ║
║  4. Upload any X-ray from:                                   ║
║     chest_xray/test/NORMAL/    (to test healthy)            ║
║     chest_xray/test/PNEUMONIA/ (to test sick)               ║
║                                                              ║
║  5. To STOP the app: press Ctrl+C in Command Prompt          ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")